In [16]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer

messages = pd.read_csv("data/data_wthout_anomailes/messages_wthout_anomailes.csv")

chat_ids = messages[['chat_id']].dropna().drop_duplicates()

messages["sent_at"] = pd.to_datetime(messages["sent_at"], errors="coerce")
# При одинаковом времени первой идёт более поздняя строка CSV.
messages = (
    messages.dropna(subset=["chat_id", "sent_at"])
    .sort_index(ascending=False)
    .sort_values("sent_at", ascending=False, kind="stable")
    .groupby("chat_id", sort=False)
    .head(5)
    .copy()
)
recent_rank = messages.groupby("chat_id", sort=False).cumcount()
messages = messages.dropna(subset=["text"]).copy()
messages["text"] = messages["text"].astype(str)

In [17]:
model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    device="cuda"
)
    
display(model.device)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2202.34it/s]


device(type='cuda', index=0)

In [18]:
# POSITIVE — признаки сделки 
# Категория	Что ищем
# purchase_intent	Намерение купить
# seller_confirmation	Подтверждение наличия товара
# price_negotiation	Обсуждение цены
# reservation	Бронирование товара
# meeting	Договорённость о встрече
# delivery	Доставка и отправка
# payment	Обсуждение оплаты
# deal_confirmation	Подтверждение завершения сделки
# item_received	Получение товара
# seller_sold	Сообщение продавца о продаже

# NEGATIVE — признаки несостоявшейся сделки
# Категория	Что ищем
# buyer_refusal	Покупатель отказался
# seller_refusal	Продавец отказался продавать
# price_disagreement	Не договорились о цене
# meeting_failure	Встреча не состоялась
# cancellation	Отмена договорённости
# item_unavailable	Товар больше недоступен
# external_sale	Продажа в другом месте
# seller_changed_mind	Продавец передумал

# NEUTRAL — слабые или фоновые признаки
# Категория	Что ищем
# general_questions	Общие вопросы
# product_condition	Обсуждение состояния товара
# product_details	Характеристики и комплектация
# contact_exchange	Обмен контактами
# availability_question	Вопрос о наличии
# uncertain_intent	Неопределённый интерес

In [19]:

sale_signals = {

    # ==================================================
    # POSITIVE: ПРИЗНАКИ ПОТЕНЦИАЛЬНОЙ И СОСТОЯВШЕЙСЯ СДЕЛКИ
    # ==================================================

    "purchase_intent": [
        "Я хочу купить этот товар",
        "Готов приобрести ваш товар",
        "Меня интересует покупка",
        "Я заинтересован в приобретении",
        "Хочу забрать этот товар",
        "Готов забрать прямо сегодня",
        "Могу купить прямо сейчас",
        "Я согласен на покупку",
        "Хочу приобрести именно этот товар",
        "Можно оформить покупку?",
        "Я беру этот товар",
        "Давайте оформим сделку",
        "Готов забрать без лишних вопросов",
        "Мне подходит ваше предложение",
        "Я готов приобрести его",
        "Хочу купить, если товар ещё доступен",
        "Договорились, я покупаю",
        "Могу приехать и забрать",
        "Готов забрать за указанную цену",
        "Я точно заинтересован в покупке",
        "Хочу забрать его себе",
        "Покупка меня интересует",
        "Давайте договоримся о покупке",
        "Готов приобрести без торга",
        "Я согласен приобрести товар"
    ],

    "seller_confirmation": [
        "Да, товар ещё продаётся",
        "Да, объявление актуально",
        "Товар в наличии",
        "Да, можете забрать",
        "Товар пока не продан",
        "Ещё не продал, можете приехать",
        "Да, всё доступно",
        "Товар свободен",
        "Можете забрать в любое удобное время",
        "Да, продаю",
        "Объявление актуально, пишите",
        "Товар пока у меня",
        "Да, можно договориться о встрече",
        "Могу продать вам",
        "Товар ещё доступен для покупки",
        "Да, готов продать",
        "Можете приезжать за ним",
        "Пока никто не забрал",
        "Да, товар не забронирован",
        "Готов встретиться для продажи"
    ],

    "price_negotiation": [
        "Можно немного уступить в цене?",
        "Сделаете небольшую скидку?",
        "Готов купить за меньшую сумму",
        "Давайте договоримся по цене",
        "Какая минимальная цена?",
        "Могу предложить другую сумму",
        "Согласны продать дешевле?",
        "Если заберу сегодня, сделаете скидку?",
        "Давайте сойдёмся на этой цене",
        "Предлагаю встретиться посередине",
        "Готов заплатить указанную сумму",
        "Цена меня устраивает",
        "Согласен на вашу цену",
        "Договорились по стоимости",
        "Хорошо, беру за эту сумму",
        "Сколько будет стоить с доставкой?",
        "Могу перевести задаток",
        "За эту цену готов забрать",
        "Давайте зафиксируем стоимость",
        "Хорошо, договорились о цене"
    ],

    "reservation": [
        "Отложите товар для меня",
        "Можно забронировать товар?",
        "Забронируйте, пожалуйста, до вечера",
        "Я точно приеду, придержите товар",
        "Можете пока никому не продавать?",
        "Готов внести предоплату за бронь",
        "Оставьте товар за мной",
        "Давайте договоримся о резерве",
        "Можно забрать завтра, придержите?",
        "Я приеду за забронированным товаром",
        "Снимите объявление, я его забираю",
        "Давайте зафиксируем договорённость",
        "Товар остаётся за мной?",
        "Подержите его до выходных",
        "Я подтверждаю бронь"
    ],

    "meeting": [
        "Договорились встретиться сегодня в шесть",
        "Я на месте, подхожу за товаром",
        "Встретимся завтра у метро, время подтверждаю",
        "Когда можно приехать за товаром?",
        "Где можно встретиться?",
        "Давайте договоримся о встрече",
        "Могу подъехать сегодня вечером",
        "Во сколько вам удобно встретиться?",
        "Напишите адрес, куда подъехать",
        "Я буду у вас через час",
        "Давайте встретимся возле метро",
        "Могу приехать завтра утром",
        "Когда вам удобно передать товар?",
        "Давайте согласуем время встречи",
        "Я уже выезжаю к вам",
        "Буду на месте через 15 минут",
        "Могу забрать товар лично",
        "Куда подъехать для покупки?",
        "Давайте встретимся сегодня",
        "Я подъехал, где вас найти?",
        "Встречаемся в оговорённом месте",
        "Подтвердите время встречи",
        "Хорошо, тогда до встречи"
    ],


    "payment": [
        "Куда перевести деньги?",
        "Как можно оплатить товар?",
        "Я готов оплатить покупку",
        "Деньги уже перевёл",
        "Оплата прошла успешно",
        "Перевёл вам указанную сумму",
        "Оплатил заказ",
        "Могу оплатить наличными",
        "Оплачу при получении",
        "Давайте согласуем способ оплаты",
        "Сколько нужно перевести?",
        "Оплата произведена полностью",
        "Вы получили перевод?",
        "Деньги отправлены на вашу карту",
        "Готов рассчитаться при встрече",
        "Я оплатил товар",
        "Получили оплату?",
        "Всё оплачено, спасибо",
        "Давайте завершим оплату",
        "Перевод отправлен, проверьте"
    ],

    "deal_confirmation": [
        "Передал товар покупателю, деньги получил",
        "Встретились и завершили сделку",
        "Спасибо, товар передан и оплачен",
        "Сделка состоялась, спасибо",
        "Спасибо за покупку",
        "Спасибо за продажу",
        "Договорённость выполнена",
        "Товар продан, спасибо",
        "Всё, сделка завершена",
        "Спасибо, всё получил",
        "Покупка успешно состоялась",
        "Деньги получил, спасибо",
        "Спасибо за оперативную сделку",
        "Договорились, товар теперь мой",
        "Спасибо, сделка завершена",
        "Покупатель забрал товар",
        "Продажа прошла успешно",
        "Спасибо, рад покупке"
    ],

    "item_received": [
        "Забрал товар, спасибо",
        "Посылку получил, всё в порядке",
        "Получила заказ, спасибо продавцу",
        "Товар получил, спасибо",
        "Всё забрал, благодарю",
        "Получил посылку",
        "Забрал заказ из пункта выдачи",
        "Товар уже у меня",
        "Всё получил в целости",
        "Посылка пришла, спасибо",
        "Получил покупку, всё отлично",
        "Забрал товар, всё хорошо",
        "Спасибо, товар доставлен",
        "Получение прошло успешно",
        "Забрал сегодня, спасибо",
        "Товар соответствует описанию",
        "Покупкой доволен",
        "Всё на месте, спасибо"
    ],


    # ==================================================
    # NEGATIVE: ПРИЗНАКИ НЕСОСТОЯВШЕЙСЯ СДЕЛКИ
    # ==================================================

    "buyer_refusal": [
        "Я передумал покупать",
        "Больше не интересует",
        "Покупать не буду",
        "Отказываюсь от покупки",
        "Мне больше не нужен этот товар",
        "Я решил не приобретать",
        "Не буду забирать товар",
        "Пока откажусь от предложения",
        "Нашёл другой вариант",
        "Покупка отменяется с моей стороны",
        "Извините, не получится купить",
        "Я больше не заинтересован",
        "Передумал, спасибо",
        "Не готов совершать покупку",
        "Спасибо, но я откажусь",
        "Уже не актуально для меня",
        "Не буду приезжать",
        "Решил пока ничего не покупать",
        "Вынужден отказаться от покупки",
        "Пожалуй, не буду брать"
    ],

    
    "seller_sold": [
        "Я уже продал этот товар",
        "Товар уже продан",
        "Продал его другому покупателю",
        "Покупатель уже забрал товар",
        "Товар больше не продаётся",
        "Уже продано, извините",
        "Продажа состоялась",
        "За товаром уже приезжали",
        "Товар забрали сегодня",
        "Уже нашёл покупателя",
        "Продал, спасибо за интерес",
        "Объявление больше не актуально, товар продан",
        "Вещь уже у нового владельца",
        "Продал через объявление",
        "Покупатель забрал его вчера",
        "Спасибо, товар уже нашёл своего покупателя"
    ],


    "seller_refusal": [
        "Я не готов продавать",
        "Не хочу продавать за такую цену",
        "Отказываюсь от сделки",
        "Не буду продавать вам товар",
        "Продажа отменяется",
        "Я решил оставить товар себе",
        "Не готов отдавать товар",
        "Не могу продать по этой цене",
        "Извините, продавать передумал",
        "Не получится продать вам",
        "Я не согласен на предложенные условия",
        "Товар продавать больше не планирую",
        "Пока не готов к продаже",
        "Решил не расставаться с вещью",
        "Не буду заключать сделку"
    ],

    "price_disagreement": [
        "Ваша цена меня не устраивает",
        "Не договорились по стоимости",
        "Слишком дорого для меня",
        "За такую сумму покупать не буду",
        "Продавец не согласился на скидку",
        "Мы не сошлись в цене",
        "Не готов платить столько",
        "Цена слишком высокая",
        "Не получается договориться о стоимости",
        "Не согласен с предложенной ценой",
        "Торг не состоялся",
        "Не договорились о скидке",
        "Предложенная сумма слишком маленькая",
        "Продавец не хочет уступать",
        "На таких условиях сделка невозможна"
    ],

    "meeting_failure": [
        "Приехал на встречу, но продавец не пришёл",
        "Покупатель не приехал за товаром",
        "Встреча не состоялась",
        "Не смог приехать за товаром",
        "Продавец не пришёл на встречу",
        "Покупатель не приехал",
        "Договорённость о встрече сорвалась",
        "Не удалось встретиться",
        "Вы не пришли в назначенное время",
        "Я не смогу подъехать",
        "Не получилось забрать товар",
        "Встречу пришлось отменить",
        "Не дождался продавца",
        "Не получилось согласовать время",
        "Встреча переносится на неопределённый срок",
        "Я не приехал за товаром",
        "Передача товара не состоялась"
    ],

    "cancellation": [
        "Встречу отменяем, сделка не состоится",
        "Давайте отменим договорённость",
        "Я отменяю заказ",
        "Покупка отменена",
        "Договорённость больше не актуальна",
        "Не будем продолжать сделку",
        "Давайте прекратим обсуждение",
        "Отмените, пожалуйста, бронь",
        "Больше не нужно резервировать товар",
        "Сделку пришлось отменить",
        "Я отменил оформление",
        "Не будем ничего оформлять",
        "Заказ отменён",
        "Решил отменить покупку",
        "Договорённость аннулируется",
        "Не получится завершить сделку"
    ],

    "item_unavailable": [
        "Товара больше нет в наличии",
        "Товар уже забрали",
        "Объявление больше не актуально",
        "Этот товар больше не продаётся",
        "Уже нет возможности купить",
        "Товар продан другому человеку",
        "Вещь уже недоступна",
        "К сожалению, товар закончился",
        "Товар больше не у меня",
        "Его уже забрали",
        "Больше не могу предложить этот товар",
        "Товар снят с продажи",
        "Продажа этого товара прекращена",
        "Товар уже передан другому покупателю",
        "Больше не принимаю предложения по этому товару"
    ],

    "external_sale": [
        "Продал товар знакомому",
        "Продал не через Авито",
        "Продал на другой площадке",
        "Продал через социальные сети",
        "Товар забрал мой знакомый",
        "Продал родственнику",
        "Нашёл покупателя самостоятельно",
        "Продал через другой сайт",
        "Договорился с покупателем вне площадки",
        "Продал человеку из своего окружения",
        "Товар купил мой коллега",
        "Продал через знакомых",
        "Покупатель нашёлся не на Авито",
        "Продал через другой сервис",
        "Уже продал вне площадки"
    ],

    "seller_changed_mind": [
        "Передумал продавать",
        "Решил оставить товар себе",
        "Больше не хочу расставаться с вещью",
        "Объявление снимаю, продавать не буду",
        "Пока оставлю товар у себя",
        "Решил не продавать",
        "Товар мне ещё нужен",
        "Продажа больше не актуальна",
        "Передумал выставлять товар на продажу",
        "Пока не планирую продавать",
        "Оставлю вещь для личного использования",
        "Решил отложить продажу",
        "Больше не рассматриваю продажу",
        "Не хочу сейчас продавать эту вещь",
        "Сниму объявление, товар оставлю себе"
    ]
}

signal_weights = {
    # POSITIVE — признаки в пользу сделки
    "purchase_intent": 0.05,
    "seller_confirmation": 0.05,
    "price_negotiation": 0.15,
    "reservation": 0.1,
    "meeting": 1.5,
    "payment": 1.0,
    "deal_confirmation": 2.5,
    "item_received": 3.0,

    # NEGATIVE — признаки против сделки
    "buyer_refusal": -1.5,
    "seller_refusal": -1.5,
    "price_disagreement": -0.7,
    "meeting_failure": -1.8,
    "cancellation": -1.8,
    "item_unavailable": -0.6,
    "external_sale": -2.5,
    "seller_changed_mind": -1.8,
    "seller_sold": -2.0,
}

signal_embeddings = {}

for category, phrases in sale_signals.items():
    signal_embeddings[category] = model.encode(
        phrases,
        normalize_embeddings=True,
        show_progress_bar=True
    )

if messages.empty:
    for category in signal_weights:
        messages[category] = pd.Series(index=messages.index, dtype=float)
else:
    message_embeddings = model.encode(
        messages["text"].tolist(),
        normalize_embeddings=True,
        batch_size=128,
        show_progress_bar=True
    )
    for category, embeddings in signal_embeddings.items():
        similarities = message_embeddings @ embeddings.T
        messages[category] = similarities.max(axis=1)


categories = list(signal_weights.keys())

# Умножаем оценки сходства на веса
weighted_scores = messages[categories].mul(signal_weights)

# Складываем веса для каждого сообщения
messages["deal_signal_score"] = weighted_scores.sum(axis=1)

# Веса пяти последних сообщений каждого чата.
recent_message_weights = {0: 3.0, 1: 2.5, 2: 2.0, 3: 1.75, 4: 1.5}
messages["recency_weight"] = recent_rank.reindex(messages.index).map(recent_message_weights)
messages["weighted_deal_signal_score"] = messages["deal_signal_score"] * messages["recency_weight"]

messages = messages.sort_values(by="weighted_deal_signal_score", ascending=False)
messages

Batches: 100%|██████████| 2147/2147 [00:48<00:00, 44.22it/s]


,chat_id,listing_id,sent_at,author,text,purchase_intent,seller_confirmation,price_negotiation,reservation,meeting,...,seller_refusal,price_disagreement,meeting_failure,cancellation,item_unavailable,external_sale,seller_changed_mind,deal_signal_score,recency_weight,weighted_deal_signal_score
200531,507236,102147,2026-08-10 13:59:07,buyer,"Спасибо, всё отлично!",0.600673,0.671593,0.590385,0.514771,0.642593,...,0.414031,0.265096,0.218491,0.255315,0.321380,0.395071,0.422146,0.601069,3.0,1.803208
76725,623378,130555,2026-09-09 08:34:20,buyer,"Спасибо, всё отлично!",0.600673,0.671593,0.590385,0.514771,0.642593,...,0.414031,0.265096,0.218491,0.255315,0.321380,0.395071,0.422146,0.601069,3.0,1.803208
331239,502621,100282,2026-08-05 22:35:01,buyer,"Спасибо, всё отлично!",0.600673,0.671593,0.590385,0.514771,0.642593,...,0.414031,0.265096,0.218491,0.255315,0.321380,0.395071,0.422146,0.601069,3.0,1.803208
147387,605043,124214,2026-09-05 19:11:24,buyer,"Спасибо, всё отлично!",0.600673,0.671593,0.590385,0.514771,0.642593,...,0.414031,0.265096,0.218491,0.255315,0.321380,0.395071,0.422146,0.601069,3.0,1.803208
178285,631590,131812,2026-09-10 15:46:11,buyer,"Спасибо, всё отлично!",0.600673,0.671593,0.590385,0.514771,0.642593,...,0.414031,0.265096,0.218491,0.255315,0.321380,0.395071,0.422146,0.601069,3.0,1.803208
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
334071,711748,148356,2026-09-25 11:06:10,seller,Не пользеумся,0.436405,0.707433,0.503172,0.552722,0.513301,...,0.688431,0.748101,0.707730,0.679333,0.750124,0.538463,0.619070,-5.332934,3.0,-15.998801
72847,602899,124695,2026-09-04 15:55:26,seller,Не пользеумся,0.436405,0.707433,0.503172,0.552722,0.513301,...,0.688431,0.748101,0.707730,0.679333,0.750124,0.538463,0.619070,-5.332934,3.0,-15.998801
33352,604095,128289,2026-09-05 10:30:21,seller,Не пользеумся,0.436405,0.707433,0.503172,0.552722,0.513301,...,0.688431,0.748101,0.707730,0.679333,0.750124,0.538463,0.619070,-5.332934,3.0,-15.998801
190478,533468,103586,2026-08-18 20:44:37,seller,Докуметнов нет,0.443198,0.715407,0.381048,0.426733,0.401681,...,0.611685,0.665524,0.618192,0.750732,0.742050,0.475643,0.511047,-5.489703,3.0,-16.469109


## Сигналы сделки по чатам

В `chat_scores` одна строка на `chat_id`. `deal_signal_score` — разность между самым сильным положительным сигналом и модулем самого сильного отрицательного сигнала по сообщениям чата. Повторение сообщения не увеличивает оценку.

Для отбора чатов на проверку таблица отсортирована по `deal_signal_strength` — силе большего из двух сигналов. Поэтому вверху могут оказаться как подтверждения, так и отказы. Если оба сигнала сильные, их разность может быть близка к нулю, поэтому обе оценки сохранены отдельно. Чаты без оценённых сообщений имеют пропуски в оценках. Это эвристика для анализа переписки.

In [23]:
weighted_message_score = messages["weighted_deal_signal_score"]

chat_evidence = (
    messages.assign(
        strongest_positive_signal=weighted_message_score.clip(lower=0),
        strongest_negative_signal=(-weighted_message_score).clip(lower=0),
    )
    .groupby("chat_id", as_index=False)
    .agg(
        scored_message_count=("deal_signal_score", "count"),
        strongest_positive_signal=("strongest_positive_signal", "max"),
        strongest_negative_signal=("strongest_negative_signal", "max"),
    )
)

chat_scores = chat_ids.merge(chat_evidence, on="chat_id", how="left", validate="one_to_one")
chat_scores["scored_message_count"] = chat_scores["scored_message_count"].fillna(0).astype(int)
chat_scores["deal_signal_score"] = (
    chat_scores["strongest_positive_signal"] - chat_scores["strongest_negative_signal"]
)
chat_scores["deal_signal_strength"] = chat_scores[
    ["strongest_positive_signal", "strongest_negative_signal"]
].max(axis=1)
chat_scores = chat_scores.sort_values(
    "deal_signal_strength", ascending=False, kind="stable"
).reset_index(drop=True)

chat_scores = chat_scores[chat_scores["scored_message_count"] > 0]
chat_scores = chat_scores[chat_scores["strongest_positive_signal"] > 0] 

chat_scores = chat_scores.sort_values("strongest_negative_signal", ascending=True)
chat_scores

,chat_id,scored_message_count,strongest_positive_signal,strongest_negative_signal,deal_signal_score,deal_signal_strength,listing_id
87280,787905,5,1.356562,2.097860,-0.741298,2.097860,152300.0
87278,575180,5,1.356562,2.097862,-0.741300,2.097862,152300.0
86854,687705,5,0.074946,2.379410,-2.304464,2.379410,155234.0
86847,745272,4,0.093682,2.392647,-2.298964,2.392647,154986.0
86846,672480,4,1.356561,2.395408,-1.038847,2.395408,154986.0
...,...,...,...,...,...,...,...
25854,816981,5,1.803208,8.262307,-6.459098,8.262307,NaN
25853,728753,5,1.356561,8.262307,-6.905746,8.262307,NaN
25102,750728,5,0.112419,8.323903,-8.211485,8.323903,122915.0
12492,537170,5,0.074946,10.036037,-9.961092,10.036037,NaN


In [ ]:
chat_scores = chat_scores[:-4]

listing_ids = messages.groupby("chat_id")["listing_id"].first()
chat_scores = chat_scores.merge(
    listing_ids,
    on="chat_id",
    how="left",
    validate="one_to_one",
)

chat_scores.to_csv("good_examples_of_verified_by_messages.csv")

In [22]:
chat_scores.median()

chat_id                      636464.000000
scored_message_count              5.000000
strongest_positive_signal         1.356562
strongest_negative_signal         6.256101
deal_signal_score                -4.850310
deal_signal_strength              6.256101
dtype: float64